# 02. 데이터 소스 조사와 선정

> 재설계 2단계 · 전체 계획은 [`plan.md`](plan.md) 참고
> 산출물: `data/api_inventory.csv`(법령 API 전체 목록), `data/source_matrix.csv`(질문 × 소스 매핑), 리포트(7절)

## 이 노트북이 결정하는 것

| 결정 | 방법 |
|---|---|
| 법령 API 195개 중 무엇을 쓰는가 | 공식 가이드를 전부 수집해 목록화(2절) → "임대차"로 전수 조회해 관련 문서 수 측정(3절) |
| 질문마다 어떤 소스가 필요한가 | 평가셋 14문항 × 후보 소스 매핑(4절) |
| 공식 검색을 쓸 수 있는가 | 지능형 법령검색(`aiSearch`)을 하네스와 **같은 평가셋·같은 지표**로 측정(5절) |
| 법령 API로 안 되는 것은 | 외부 공공 소스 조사(6절) |

## 1. 조사 방법

LegalBench-RAG(Pipitone & Alami, 2024)는 법률 RAG 실패의 상당수가 **근거 문서가 코퍼스에 없거나 검색 단위가 맞지 않아서** 생긴다고 본다.
그래서 모델을 고르기 전에 "정답 근거가 어느 소스에 있는가"부터 확인한다.

1. **전수 목록화** — 사람의 기억이나 요약에 의존하지 않고, 공식 가이드 페이지 195개를 모두 받아 요청 주소·target 코드를 추출한다.
   (웹 요약 도구로 먼저 조회했을 때 target 코드가 일부 틀리게 나와, 원문 파싱으로 바꿨다.)
2. **관련성 측정** — 목록 API마다 같은 검색어로 조회해 문서 수를 센다. 수가 0이면 제외 후보.
3. **질문 × 소스 매핑** — 평가 질문마다 후보 소스에서 결과가 나오는지 확인한다.
4. **외부 소스** — 법령 API로 답할 수 없는 질문(Q10 정책)은 공공데이터포털 등을 조사한다.

API 응답은 하네스와 같은 디스크 캐시(`~/.cache/aigo-ai/api`)를 쓰므로, 재실행해도 API를 다시 부르지 않는다.

In [1]:
import html
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
  ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "harness"))
import prepare as P  # 하네스의 캐시된 API 호출·링크 규칙 재사용

HERE = ROOT / "notebooks" / "redesign"
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)
GUIDE_CACHE = P.CACHE_DIR / "guides"
GUIDE_CACHE.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

## 2. 법령 API 전체 목록

공동활용 사이트의 가이드 목록(`guideList.do`)에서 항목 ID를 뽑고, 항목마다 상세 페이지(`guideResult.do`)를 받아 요청 주소를 파싱한다.
가이드 페이지는 공개 문서라 인증키가 필요 없다.

In [2]:
GUIDE_BASE = "https://open.law.go.kr/LSO/openApi"


def _page_text(raw: str) -> str:
  t = re.sub(r"<script[\s\S]*?</script>|<style[\s\S]*?</style>", "", raw)
  return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", t)))


def guide_page(guide_id: str) -> str:
  path = GUIDE_CACHE / f"{guide_id}.html"
  if not path.exists():
    resp = requests.post(f"{GUIDE_BASE}/guideResult.do", data={"htmlName": guide_id}, timeout=30)
    path.write_text(resp.text, encoding="utf-8")
    time.sleep(0.4)
  return path.read_text(encoding="utf-8")


list_path = GUIDE_CACHE / "_guideList.html"
if not list_path.exists():
  list_path.write_text(requests.get(f"{GUIDE_BASE}/guideList.do", timeout=30).text, encoding="utf-8")
guides = re.findall(r"openApiGuide\('([^']+)'\)[^>]*>\s*([^<]+)<", list_path.read_text(encoding="utf-8"))


def category(name: str, guide_id: str) -> str:
  rules = [
    ("모바일", lambda: guide_id.startswith("mob")),
    ("맞춤형", lambda: "맞춤형" in name),
    ("지식베이스", lambda: any(k in name for k in ["용어", "지능형", "관련법령"])),
    ("부처 법령해석", lambda: guide_id.startswith("cgmExpc")),
    ("위원회 결정문", lambda: "위원회" in name or "심판원" in name or "사전컨설팅" in name),
    ("별표·서식", lambda: "별표" in name),
    ("판례·결정례·해석례", lambda: any(k in name for k in ["판례", "결정례", "해석례", "심판례"])),
    ("행정규칙", lambda: "행정규칙" in name),
    ("자치법규", lambda: "자치법규" in name),
    ("조약·학칙", lambda: any(k in name for k in ["조약", "학칙"])),
  ]
  return next((c for c, f in rules if f()), "법령")


rows = []
for gid, name in guides:
  text = _page_text(guide_page(gid))
  url = re.search(r"요청 URL\s*:\s*(\S+)", text).group(1)
  target = re.search(r"target=([A-Za-z]+)", url)
  rows.append({
    "guide_id": gid,
    "name": name.strip(),
    "category": category(name, gid),
    "endpoint": url.split("?")[0].rsplit("/", 1)[-1],
    "target": target.group(1) if target else "",
  })

inventory = pd.DataFrame(rows)
inventory.to_csv(DATA / "api_inventory.csv", index=False, encoding="utf-8-sig")
print(f"가이드 {len(inventory)}개, target 코드 {inventory['target'].nunique()}종")
inventory.groupby("category").agg(n=("guide_id", "count"), 예시=("name", lambda s: " / ".join(s[:3])))

가이드 195개, target 코드 99종


,n,예시
category,,
맞춤형,6,맞춤형 법령 목록 조회 / 맞춤형 법령 조문 목록 조회 / 맞춤형 행정규칙 목록 조회
모바일,20,법령 목록 조회 / 법령 본문 조회 / 행정규칙 목록 조회
법령,24,현행법령(시행일) 목록 조회 / 현행법령(시행일) 본문 조회 / 현행법령(공포일) 목록 조회
별표·서식,3,법령 별표ㆍ서식 목록 조회 / 행정규칙 별표ㆍ서식 목록 조회 / 자치법규 별표ㆍ서식 목록 조회
부처 법령해석,76,고용노동부 법령해석 목록 조회 / 고용노동부 법령해석 본문 조회 / 국토교통부 법령해석 목록 조회
위원회 결정문,34,개인정보보호위원회 결정문 목록 조회 / 개인정보보호위원회 결정문 본문 조회 / 고용보험심사위원회 결정문 목록 조회
자치법규,5,법령 기준 자치법규 연계 관련 목록 조회 / 법령-자치법규 연계현황 조회 / 자치법규 목록 조회
조약·학칙,4,조약 목록 조회 / 조약 본문 조회 / 학칙ㆍ공단ㆍ공공기관 목록 조회
지식베이스,11,법령 용어 목록 조회 / 법령 용어 본문 조회 / 법령용어 조회


## 3. 관련성 측정: "임대차"로 전수 조회

목록 API(`lawSearch.do`) 중 모바일·맞춤형(별도 신청 필요)을 뺀 target마다 `query=임대차`로 1건만 조회해 **전체 건수**를 기록한다.
응답 구조가 target마다 달라서 `totalCnt`(또는 `검색결과개수`)를 재귀적으로 찾는다.

In [3]:
def find_count(obj) -> int | None:
  if isinstance(obj, dict):
    for k in ("totalCnt", "검색결과개수"):
      if k in obj and str(obj[k]).isdigit():
        return int(obj[k])
    for v in obj.values():
      n = find_count(v)
      if n is not None:
        return n
  return None


PROBE_CACHE = P.CACHE_DIR / "probe"
PROBE_CACHE.mkdir(exist_ok=True)


def count(target: str, query: str, **params) -> int | str:
  """전체 건수. API 신청이 안 된 소스는 '미신청', 그 밖의 실패는 '오류'. 실패도 캐시해 재실행을 빠르게 한다."""
  key = re.sub(r"[^\w=-]", "_", f"{target}-{query}-{sorted(params.items())}")
  path = PROBE_CACHE / f"{key}.json"
  if path.exists():
    return json.loads(path.read_text(encoding="utf-8"))["value"]
  try:
    value = find_count(P._api(P._SEARCH_URL, target=target, query=query, display=1, **params))
    value = 0 if value is None else value
  except Exception:
    resp = requests.get(P._SEARCH_URL, params={"OC": P._oc(), "type": "JSON", "target": target, "query": query}, timeout=60)
    value = "미신청" if "미신청" in resp.text else "오류"
  path.write_text(json.dumps({"value": value}, ensure_ascii=False), encoding="utf-8")
  return value


probe = (
  inventory[(inventory.endpoint == "lawSearch.do") & ~inventory.category.isin(["모바일", "맞춤형"])]
  .drop_duplicates("target")
  .copy()
)
probe["임대차"] = [count(t, "임대차") for t in probe.target]
probe["임대차_본문"] = [count(t, "임대차", search=2) if c in ("판례·결정례·해석례", "법령") else None
                    for t, c in zip(probe.target, probe.category)]
status = probe["임대차"].map(lambda v: v if isinstance(v, str) else ("결과 있음" if v > 0 else "0건"))
print(f"조회 {len(probe)}종:", status.value_counts().to_dict())
relevant = probe[status == "결과 있음"].sort_values("임대차", key=lambda s: s.astype(int), ascending=False)
relevant[["category", "name", "target", "임대차", "임대차_본문"]]

조회 89종: {'미신청': 34, '0건': 30, '결과 있음': 23, '오류': 2}


,category,name,target,임대차,임대차_본문
23,법령,삭제 데이터 목록 조회,delHst,13901,13901
22,법령,법률명 약칭 조회,lsAbrv,2714,2714
33,판례·결정례·해석례,판례 목록 조회,prec,373,7341
185,위원회 결정문,조세심판원,ttSpecialDecc,138,None
0,법령,현행법령(시행일) 목록 조회,eflaw,136,12424
101,지식베이스,일상용어 조회,dlytrm,71,None
100,지식베이스,법령용어 조회,lstrmAI,61,None
111,부처 법령해석,국토교통부 법령해석 목록 조회,molitCgmExpc,31,None
72,지식베이스,법령 용어 목록 조회,lstrm,31,None
67,별표·서식,법령 별표ㆍ서식 목록 조회,licbyl,28,None


## 4. 질문 × 소스 매핑

평가 질문마다 **핵심어**로 후보 소스를 조회한다. 핵심어는 사람이 질문에서 뽑은 것이라 편향이 있을 수 있어, 표에 그대로 남긴다.
법령 조문은 자연어 질문을 그대로 받는 지능형 검색(`aiSearch`)으로 조회하고, 정답 근거가 몇 위에 나오는지도 본다.

In [4]:
# API 신청이 안 되어 조회하지 못한 소스 (OPEN API 신청 화면에서 법령종류를 체크하면 열린다)
probe[status == "미신청"].groupby("category")["name"].apply(lambda s: ", ".join(s.str.replace(" 목록 조회", "")))

category
법령                                                                                      영문 법령
별표·서식                                                                  행정규칙 별표ㆍ서식, 자치법규 별표ㆍ서식
부처 법령해석       고용노동부 법령해석, 재정경제부 법령해석, 해양수산부 법령해석, 행정안전부 법령해석, 기후에너지환경부 법령해석, 관세청 법령해석, 국세청...
위원회 결정문       개인정보보호위원회 결정문, 고용보험심사위원회 결정문, 공정거래위원회 결정문, 금융위원회 결정문, 노동위원회 결정문, 방송미디어통신위원회,...
자치법규                                                                   자치법규, 자치법규 기준 법령 연계 관련
조약·학칙                                                                          조약, 학칙ㆍ공단ㆍ공공기관
판례·결정례·해석례                                                                       헌재결정례, 행정심판례
행정규칙                                                                        행정규칙, 행정규칙 신구법 비교
Name: name, dtype: str

In [5]:
eval_items = {it["id"]: it for it in json.loads(P.EVAL_PATH.read_text(encoding="utf-8"))["items"]}

KEYWORDS = {  # 질문 → 소스 조회용 핵심어 (사람이 작성)
  "Q01": "임대차 해지", "Q02": "전세권설정", "Q04": "전세자금대출", "Q05": "원상복구",
  "Q06": "차임", "Q07": "층간소음", "Q09": "가계약금", "Q10": "청년 월세",
  "Q11": "주거침입", "Q12": "임대차 신고", "T01": "계약갱신요구", "T02": "임대차 신고",
}
SOURCES = {  # 표시명 → (target, 추가 파라미터)
  "판례(본문)": ("prec", {"search": 2}),
  "법령해석례(본문)": ("expc", {"search": 2}),
  "국토부 해석": ("molitCgmExpc", {}),
  "행정심판례": ("decc", {}),
  "행정규칙": ("admrul", {}),
  "자치법규": ("ordin", {}),
  "법령 별표·서식": ("licbyl", {}),
  "법령용어": ("lstrm", {}),
  "일상용어": ("dlytrm", {}),
}


def ai_search(query: str, n: int = 20) -> list[dict]:
  res = P._api(P._SEARCH_URL, target="aiSearch", search=0, query=query, display=n)["aiSearch"]
  return P._as_list(res.get("법령조문"))


def gold_rank(item: dict, hits: list[dict]) -> int | None:
  gold = {f"{s['law_name']}|{P.jo_to_label(s['jo'])}" for s in item["gold_sources"] if s["type"] == "law" and s.get("jo")}
  keys = [f"{h['법령명']}|{P.article_label(h['조문번호'], h.get('조문가지번호'))}" for h in hits]
  return next((i + 1 for i, k in enumerate(keys) if k in gold), None)


matrix = []
for qid, kw in KEYWORDS.items():
  item = eval_items[qid]
  hits = ai_search(item["question"])
  row = {
    "id": qid,
    "핵심어": kw,
    "조문(aiSearch) 정답 순위": gold_rank(item, hits),
    "aiSearch 1위": f"{hits[0]['법령명']} {P.article_label(hits[0]['조문번호'], hits[0].get('조문가지번호'))}" if hits else None,
  }
  for label, (target, params) in SOURCES.items():
    row[label] = count(target, kw, **params)
  matrix.append(row)

source_matrix = pd.DataFrame(matrix).set_index("id")
source_matrix.to_csv(DATA / "source_matrix.csv", encoding="utf-8-sig")
source_matrix

,핵심어,조문(aiSearch) 정답 순위,aiSearch 1위,판례(본문),법령해석례(본문),국토부 해석,행정심판례,행정규칙,자치법규,법령 별표·서식,법령용어,일상용어
id,,,,,,,,,,,,
Q01,임대차 해지,NaN,상속세 및 증여세법 시행규칙 제16조의2,1153,268,2,5,0,0,2,0,0
Q02,전세권설정,NaN,주택임대차보호법 시행령 제1조,757,24,1,0,3,1,0,0,0
Q04,전세자금대출,NaN,종합부동산세법 시행령 제3조,209,0,0,0,0,31,0,0,2
Q05,원상복구,NaN,동물보호법 제81조,603,30,12,48,0,2,8,3,14
Q06,차임,1.0,민법 제618조,2028,14,1,0,1,3,1,4,14
Q07,층간소음,3.0,공동주택 층간소음의 범위와 기준에 관한 규칙 제1조,13,4,1,0,2,116,1,2,7
Q09,가계약금,NaN,민법 제465조,119,0,0,0,1,0,0,2,0
Q10,청년 월세,NaN,주거기본법 제13조,3,9,0,1,0,8,0,0,0
Q11,주거침입,NaN,지방세특례제한법 시행령 제17조의3,432,0,0,0,0,0,0,2,3


### 4-1. 용어 질문(Q6): 법령용어 ↔ 일상용어 연계

1단계에서 법령용어 API의 "차임"은 한영사전 항목("rent")만 나왔다. 법령용어-일상용어 연계 API(`lstrmRlt`)는
법령용어와 일상용어를 동의어·상위어·연관어로 이어 준다. 사회초년생의 말("월세")과 법률 용어("차임")를 잇는 데 쓸 수 있는지 확인한다.

In [6]:
def show(obj, limit=900):
  """응답 속 상세링크의 인증키(OC 값)는 가려서 출력한다."""
  print(re.sub(r"OC=[^&\"]+", "OC=***", json.dumps(obj, ensure_ascii=False))[:limit])


for term in ["차임", "보증금"]:
  print(f"■ 법령용어 '{term}' → 일상용어")
  show(P._api(P._SERVICE_URL, target="lstrmRlt", query=term))
print("■ 일상용어 '월세' → 법령용어")
show(P._api(P._SERVICE_URL, target="dlytrmRlt", query="월세"))

■ 법령용어 '차임' → 일상용어
{"lstrmRltService": {"검색결과개수": "1", "키워드": "차임", "target": "lstrmRlt", "법령용어": {"id": "1", "연계용어": [{"용어간관계링크": "/DRF/lawService.do?OC=***&target=dlytrmRlt&type=XML&MST=96675", "id": "1", "일상용어명": "깔세", "용어관계": "동의어", "용어관계코드": "140301", "일상용어조회링크": "/DRF/lawSearch.do?OC=***&target=dlytrm&type=XML&query=깔세"}, {"용어간관계링크": "/DRF/lawService.do?OC=***&target=dlytrmRlt&type=XML&MST=96673", "id": "2", "일상용어명": "년세", "용어관계": "동의어", "용어관계코드": "140301", "일상용어조회링크": "/DRF/lawSearch.do?OC=***&target=dlytrm&type=XML&query=년세"}, {"용어간관계링크": "/DRF/lawService.do?OC=***&target=dlytrmRlt&type=XML&MST=4381", "id": "3", "일상용어명": "대가", "용어관계": "상위어", "용어관계코드": "140303", "일상용어조회링크": "/DRF/lawSearch.do?OC=***&target=dlytrm&type=XML&query=대가"}, {"용어간관계링크": "/DRF/lawService.do?OC=***&target=dlytrmRlt&type=XML&MST=21609", "id": "4", "일상용어명": "대임", "용어관계": "반의어", "용어관계코드": "140302", "일상용어조회링크": "/DRF/lawSearch.do
■ 법령용어 '보증금' → 일상용어
{"lstrmRltService": {"검색결과개수": "1", "키워드": "보증금", "target": 

### 4-2. 시점 질문: 조문별 변경 이력

하네스 코퍼스(v1)는 시행일 판본을 **전부** 받아 조문 텍스트를 비교했다(판본 수 × 전체 조문).
조문별 변경 이력 API(`lsJoHstInf`)는 특정 조문이 바뀐 시점만 알려 주므로, 필요한 판본만 받는 방식으로 수집 비용을 줄일 수 있는지 확인한다.

In [7]:
hist = P._api(P._SERVICE_URL, target="lsJoHstInf", ID="001248", JO="000603", display=100)
show(hist, 1500)

{"LawService": {"law": {"id": "1", "조문정보": {"조문번호": "000603", "조문링크": "/DRF/lawService.do?OC=***&target=eflaw&MST=220619&JO=000603&efYd=20200731&type=HTML", "변경사유": "본조신설", "조문변경일": "20200731"}, "법령정보": {"법령일련번호": "220619", "시행일자": "20200731", "법령구분명": "법률", "소관부처명": "법무부,국토교통부", "공포번호": "17470", "제개정구분명": "일부개정", "공포일자": "20200731", "소관부처코드": "1270000,1613000"}}, "법령ID": "001248", "법령명한글": "주택임대차보호법", "target": "lsJoHstInf", "totalCnt": "1"}}


## 5. 공식 지능형 법령검색을 하네스로 측정

`aiSearch`는 자연어 질문을 받아 **조문 단위**로 결과를 준다. 하네스 평가 함수(`prepare.evaluate`)에 그대로 넣어
BM25 기준선(dev recall@5 = 0.148)과 같은 조건으로 비교한다. 판례·해석례는 반환하지 않으므로 그 몫은 놓친다.

In [8]:
def ai_retrieve(query: str) -> list[dict]:
  out = []
  for h in ai_search(query):
    label = P.article_label(h["조문번호"], h.get("조문가지번호"))
    key = f"{h['법령명']}|{label}"
    out.append({"key": key, "version_key": f"{key}@{P._iso(h['시행일자'][:8])}", "url": P.law_url(h["법령명"], label)})
  return out


compare = pd.DataFrame({
  "BM25 기준선 (harness/rag.py)": {"dev recall@5": 0.148, "dev mrr@5": 0.148, "test recall@5": 0.091},
  "지능형 법령검색 (aiSearch)": {
    "dev recall@5": P.evaluate(ai_retrieve, "dev")["recall@5"],
    "dev mrr@5": P.evaluate(ai_retrieve, "dev")["mrr@5"],
    "test recall@5": P.evaluate(ai_retrieve, "test")["recall@5"],
  },
}).round(3)
compare

,BM25 기준선 (harness/rag.py),지능형 법령검색 (aiSearch)
dev recall@5,0.148,0.259
dev mrr@5,0.148,0.120
test recall@5,0.091,0.182


## 6. 외부 공공 소스 (법령 API 밖)

Q10(청년 주거비 지원)은 법령이 아니라 **정책 정보**가 필요하다. 또 법령 조문은 사회초년생에게 어렵기 때문에, 쉬운 설명 자료가 있으면 답변 품질에 도움이 된다.
공공데이터포털과 기관 사이트를 조사했다(2026-09-30 기준, 각 포털 안내 페이지 확인).

| 소스 | 제공 기관 | 내용 | 승인 | 라이선스 | 쓰임 |
|---|---|---|---|---|---|
| [온통청년 청년정책 API](https://www.data.go.kr/data/15143273/openapi.do) | 한국고용정보원 | 청년정책(일자리·**주거**·금융·복지·교육) 이름·설명·기관·신청 링크 | 개발 자동 / 운영 심의 | 제한 없음 | **Q10 정책 질문** |
| [중앙부처 복지서비스 API](https://www.data.go.kr/data/15090532/openapi.do) | 한국사회보장정보원(복지로) | 복지서비스 목록·지원 대상·선정 기준·신청 방법 | 자동 | 제한 없음 | Q10 보조 (청년 외 주거급여 등) |
| [생활법령 백문백답 API](https://www.data.go.kr/data/15000335/openapi.do) | 법제처 | 주택임대차 등 12개 분야 생활법률 **질의응답** | 심의 | 출처표시(KOGL 1유형) | 쉬운 말 설명, 질문 재표현 |
| [생활법령정보 조회 API](https://www.data.go.kr/data/15000215/openapi.do) | 법제처(찾기쉬운 생활법령정보) | 생활법령 분류·관련 판례·행정심판 사례 | 심의 | 출처표시(KOGL 1유형) | 주제별 설명 |

- 네 소스 모두 **공공데이터포털(또는 온통청년) 인증키가 필요**하다. 개발 계정은 하루 약 100건으로 제한된다.
- 법제처 두 API는 SOAP(XML) 방식이라 파서가 따로 필요하다.
- 아래 셀은 `.env`에 키가 있을 때만 실행된다(없으면 건너뜀).

In [9]:
from dotenv import dotenv_values

env = dotenv_values(ROOT / ".env")
external_keys = {name: bool(env.get(name)) for name in ["DATA_GO_KR_KEY", "YOUTHCENTER_KEY"]}
print("외부 소스 인증키:", external_keys)
if not any(external_keys.values()):
  print("→ 키가 없어 외부 소스 조회를 건너뜁니다. 신청 후 .env 에 추가하고 다시 실행하세요.")

외부 소스 인증키: {'DATA_GO_KR_KEY': False, 'YOUTHCENTER_KEY': False}
→ 키가 없어 외부 소스 조회를 건너뜁니다. 신청 후 .env 에 추가하고 다시 실행하세요.


## 7. 결과 리포트

### 7-1. 발견

1. **법령 API는 195개 가이드, 99종 target**으로 구성된다. 모바일·맞춤형·부처별 해석을 빼면 실제로 볼 소스는 훨씬 적다.
2. **판례는 이름 검색만으로는 거의 잡히지 않는다.** "임대차"로 사건명을 검색하면 373건이지만 본문 검색은 **7,341건**이다.
   하네스 코퍼스 v1(판례 342건)은 본문 기준 후보의 약 5%만 담고 있었다. 법령해석례도 이름 23건 대 본문 180건으로 같은 양상이다.
3. **국토교통부 법령해석(`molitCgmExpc`)이 따로 있다.** "임대차" 31건, "계약갱신요구" 21건으로, 법제처 해석례에는 없는 실무 해석이 담겨 있다.
4. **공식 지능형 검색(`aiSearch`)은 쓸 만하지만 단독으로는 부족하다.**
   - 하네스 dev 기준 recall@5 **0.259**로 BM25 기준선(0.148)보다 높지만, mrr@5는 0.120으로 오히려 낮다. 정답을 찾더라도 순위가 낮다.
   - 원문 질문 12개 중 정답 조문이 20위 안에 든 것은 4개(Q6·Q7·T01·T02)뿐이다. 구어체의 긴 질문에서는 1위가 엉뚱한 법령인 경우가 많다(예: Q1 → 상속세 및 증여세법 시행규칙).
   - Q12("임대차 계약 신고 기한")에서는 **민간임대주택법의 임대차계약 신고(3개월 이내)**가 1위로 나온다. 일반 임차인에게 적용되는 부동산거래신고법(30일 이내)과 헷갈리기 쉬운 **함정 문서**라, 생성 단계에서 적용 대상을 구분하는지 평가해야 한다.
   → 코퍼스가 아니라 **검색 후보를 넓히는 부품**(하이브리드의 한 축)으로 4단계에서 실험한다.
5. **용어 연계 API는 질의 확장 사전으로 쓸 수 있다.** 법령용어 "차임"은 일상용어 깔세·년세(동의어), 대가(상위어)로 이어지고,
   일상용어 "월세"는 법령용어 월세금액·월임대료(동의어)로 이어진다. 사용자의 일상어를 법률 용어로 바꾸는 데 쓴다.
   (출력 상위에는 "월세 → 차임" 직접 연결이 보이지 않아, 연결이 부족한 부분은 수동 사전으로 보완할 수 있다.)
6. **조문별 변경 이력 API(`lsJoHstInf`)로 시점별 수집 비용을 줄일 수 있다.** 주택임대차보호법 제6조의3은 "2020-07-31 본조신설" 한 건만 나온다.
   v1처럼 모든 시행일 판본을 받지 않고, **바뀐 날짜의 판본만** 받으면 된다.
7. **API 신청 범위가 좁다.** 조회한 89종 중 **34종이 "미신청된 목록/본문"**으로 막혀 있다(행정규칙·자치법규·행정심판례·헌재결정례 등). 결과가 있는 소스는 23종, 0건 30종, 기타 오류 2종이다. 자치법규에는 지자체 **청년 월세 지원 조례**가 있어 Q10에 필요하다.

### 7-2. 코퍼스 v2 소스 결정

| 소스 | 결정 | 근거 |
|---|---|---|
| 법령 조문 (현행 + 변경 이력 판본) | **채택** | 모든 질문의 1차 근거. 판본은 `lsJoHstInf`로 변경일만 수집 |
| 판례 (본문 검색) | **채택** | 이름 검색의 약 20배. 7,341건 중 관련 판례만 거르는 규칙은 3단계에서 설계 |
| 법령해석례 (법제처, 본문 검색) | **채택** | 180건. 절차 질문(Q12)의 근거 |
| 국토교통부 법령해석 | **채택** | 계약갱신·임대차 신고 등 실무 해석 |
| 법령용어 · 일상용어 연계 | **채택 (사전)** | 검색 인덱스가 아니라 질의 확장 사전으로 사용 |
| 지능형 법령검색 `aiSearch` | **실험** | 4단계에서 BM25·dense와 결합하는 후보 생성기로 비교 |
| 법령 별표·서식 | 보류 | 28건. 표준계약서 서식이 있는지 3단계에서 확인 |
| 행정규칙 · 자치법규 · 행정심판례 | **신청 후 재조사** | 현재 미신청. 자치법규는 Q10(청년 월세 조례)에 필요 |
| 온통청년 청년정책 · 복지로 | **채택 예정 (키 필요)** | Q10 정책 질문. 개발 단계 자동 승인 |
| 생활법령 백문백답 · 생활법령정보 | **채택 예정 (키 필요)** | 쉬운 말 설명과 질의응답. 심의 승인이라 시간이 걸림 |
| 위원회 결정문 · 부처별 해석(국토부 외) · 조약 등 | 제외 | "임대차" 결과가 없거나 주제와 무관 (조세심판원 138건은 임대소득 과세 사건이라 제외) |

### 7-3. 사람이 해야 할 일

1. **법령 OPEN API 신청 범위 확대**: open.law.go.kr → OPEN API 신청 → 등록된 API에서 행정규칙·자치법규·행정심판례 등 법령종류 체크
2. **외부 소스 인증키 신청**: 공공데이터포털(`DATA_GO_KR_KEY` — 복지로, 생활법령), 온통청년(`YOUTHCENTER_KEY`)
3. 키를 받으면 `.env`에 넣고 이 노트북을 다시 실행해 6절과 7-2를 갱신한다.

### 7-4. 한계

- 관련성 측정은 "임대차" 한 단어의 건수라, 관련 문서의 **양**은 보여 주지만 **질**은 보여 주지 않는다.
- 질문 × 소스 매핑의 핵심어는 사람이 골랐다. 다른 핵심어를 쓰면 건수가 달라질 수 있다.
- `delHst`(삭제 데이터), `lsAbrv`(약칭)은 검색어와 관계없이 전체 건수를 돌려주는 것으로 보여 판단에서 뺐다.
- 외부 소스 4종은 인증키가 없어 실제 응답을 확인하지 못했다. 포털 안내 페이지 기준이다.